# Data API Collector — Databricks Quick Start

Verify connectivity to all services and run a quick end-to-end test.

**For deep-dive examples by capability, see:**
- [`examples/kafka_streaming.ipynb`](examples/kafka_streaming.ipynb) — 9 Kafka streaming use cases (core + SLED)
- [`examples/neo4j_graph.ipynb`](examples/neo4j_graph.ipynb) — Populate and query SLED graph data
- [`examples/postgres_jdbc.ipynb`](examples/postgres_jdbc.ipynb) — Populate and read SLED relational tables

All notebooks share configuration from [`examples/_config.ipynb`](examples/_config.ipynb) — update HOST and secret scope there once.

In [ ]:
%pip install neo4j
dbutils.library.restartPython()

In [ ]:
%run "./examples/_config"

In [ ]:
import neo4j

---
## 1. Service Health Checks

In [ ]:
checks = {
    "PostgreSQL ORM":  "/data-sources/orm",
    "PostgreSQL SQL":  "/data-sources/raw/sql",
    "Neo4j Health":    "/data-sources/neo4j/health",
    "Neo4j Version":   "/data-sources/neo4j/version",
    "Redis":           "/redis/test",
    "Kafka Events":    "/kafka/events?limit=1",
}

for name, path in checks.items():
    try:
        r = requests.get(f"{API_URL}{path}", headers=HEADERS, timeout=10)
        status = r.json().get("status", r.status_code)
        print(f"  {name}: {status}")
    except Exception as e:
        print(f"  {name}: FAILED — {e}")

---
## 2. Quick Kafka Test

In [ ]:
resp = requests.post(f"{API_URL}/kafka/generators/start", headers=HEADERS, json={
    "use_case": "fraud_detection",
    "rows_per_batch": 100,
    "batch_interval_seconds": 1.0,
    "timeout_minutes": 5,
})
print(resp.json())

In [ ]:
fraud_schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("user_id", IntegerType()),
    StructField("merchant_id", IntegerType()),
    StructField("amount", DecimalType(10, 2)),
    StructField("currency", StringType()),
    StructField("merchant_category", StringType()),
    StructField("payment_method", StringType()),
    StructField("ip_address", StringType()),
    StructField("device_id", StringType()),
    StructField("latitude", DecimalType(8, 5)),
    StructField("longitude", DecimalType(9, 5)),
    StructField("card_type", StringType()),
    StructField("event_timestamp", StringType()),
])

fraud_df = read_kafka_stream("streaming-fraud-transactions", fraud_schema)

dbutils.fs.rm(f"{CHECKPOINT_BASE}/quickstart_fraud", recurse=True)
display(fraud_df, checkpointLocation=f"{CHECKPOINT_BASE}/quickstart_fraud")

---
## 3. Quick Neo4j Test

In [ ]:
driver = neo4j.GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session() as session:
    result = session.run("RETURN 1 AS test")
    print(f"Neo4j connected: {result.single()['test']}")

    result = session.run("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS count ORDER BY count DESC")
    records = [record.data() for record in result]
    if records:
        display(spark.createDataFrame(records))
    else:
        print("No nodes yet — run examples/neo4j_graph.ipynb to populate SLED data")

driver.close()

---
## 4. Quick PostgreSQL Test

In [ ]:
df = read_pg_table("kafka_event_logs")
print(f"Kafka event logs: {df.count()} rows")
display(df.limit(10))

---
## 5. Cleanup

In [ ]:
generators = requests.get(f"{API_URL}/kafka/generators", headers=HEADERS).json()
for g in generators:
    if g["status"] == "running":
        requests.post(f"{API_URL}/kafka/generators/{g['generator_id']}/stop", headers=HEADERS)
        print(f"Stopped {g['use_case']} generator {g['generator_id']}")

cleanup = requests.delete(f"{API_URL}/kafka/generators/cleanup", headers=HEADERS).json()
print(f"Cleaned up {cleanup.get('removed', 0)} generator(s)")